Lab 1: HyperNetX Basics (Hypergraph, Hyperedges, Incidence, Dual)
Goal: learn basic objects and methods only (no contagion/advanced analytics).

In [2]:
import pandas as pd
from hypernetx import Hypergraph

def build_H_from_edge_sets(E):
    """
    PAPER VIEW:
      Input is a hypergraph H = (V, 𝓔) given by its hyperedge set 𝓔.
      We represent 𝓔 as a dictionary:
          E = { edge_id : set_of_vertices }
      so each key is a hyperedge e ∈ 𝓔, and E[edge_id] is its vertex set.

    HYPERNETX VIEW:
      HyperNetX can store the incidence relation "v ∈ e" as a table with one row per incidence.
      This avoids a constructor path that sometimes errors depending on versions.
    """
    rows = []
    for e_id, vertices in E.items():     # e_id corresponds to a hyperedge e ∈ 𝓔(H)
        for v in vertices:              # v corresponds to a vertex v ∈ V(H)
            rows.append({
                "edges": e_id,          # identifies the hyperedge e
                "nodes": v,             # identifies the vertex v
                "weight": 1,            # not used in our paper, but required column for this HyperNetX constructor path
                "misc_properties": None # required column for this HyperNetX constructor path
            })
    incidence_df = pd.DataFrame(rows)

    # This builds H from the incidence relation {(e,v): v ∈ e}.
    return Hypergraph(incidence_df)

In [5]:
# Defining a sample hypergraph with hyperedges e1, e2, e3, e4 and vertices a, b, c, d, e, f.
# PAPER: Define hyperedges 𝓔(H). The vertex set V(H) is the union of these.
E = {
    "e1": {"a", "b", "c", "d"},  # hyperedge e1
    "e2": {"b", "c"},            # hyperedge e2
    "e3": {"c", "d", "e"},       # hyperedge e3
    "e4": {"e", "f"},            # hyperedge e4
}

H = build_H_from_edge_sets(E)
H

None hypernetx.classes.hypergraph.Hypergraph

In [ ]:
def print_hyperedges(H, title="Hyperedges 𝓔(H)"):
    """
    PAPER: Prints each hyperedge e ∈ 𝓔(H) as a subset of V(H).
    """
    print(title)
    for e_id, vertices in H.incidence_dict.items():  # edge -> vertices
        print(f"  {e_id}: {sorted(vertices)}")

print_hyperedges(H)

Hyperedges 𝓔(H)
  e1: ['a', 'b', 'c', 'd']
  e2: ['b', 'c']
  e3: ['c', 'd', 'e']
  e4: ['e', 'f']


In [10]:
# PAPER: V(H) is the union of all hyperedges.

V = set().union(*H.incidence_dict.values()) 
# what is happening above - union of all the sets of vertices in the hyperedges so we get the unique vertices in the hypergraph
E_ids = list(H.incidence_dict.keys())
# what is happening above - we are getting the list of hyperedge ids from the incidence dictionary of the hypergraph

print("H incidence dictionary view - ", H.incidence_dict)  # This is the internal representation of the hypergraph in HyperNetX.
print("|V(H)| =", len(V))
print("|𝓔(H)| =", len(E_ids))

# PAPER: size of a hyperedge e is |e|.
edge_size = {e: len(set(verts)) for e, verts in H.incidence_dict.items()} 
# create a dictionary where keys are hyperedge ids
# and values are the size of the hyperedge (number of unique vertices in it)
print("Hyperedge sizes |e|:", edge_size)

# PAPER: degree of a vertex v is the number of hyperedges containing it:
# deg_H(v) = |{ e ∈ 𝓔(H) : v ∈ e }|
deg = {}
for e, verts in H.incidence_dict.items():
    for v in verts:
        deg[v] = deg.get(v, 0) + 1 # for each vertex v in the hyperedge e, we increment its degree count by 1 because it is contained in at least one hyperedge (e)

print("Vertex degrees deg_H(v):", dict(sorted(deg.items())))

H incidence dictionary view -  {'e1': ['d', 'a', 'b', 'c'], 'e2': ['b', 'c'], 'e3': ['d', 'e', 'c'], 'e4': ['f', 'e']}
|V(H)| = 6
|𝓔(H)| = 4
Hyperedge sizes |e|: {'e1': 4, 'e2': 2, 'e3': 3, 'e4': 2}
Vertex degrees deg_H(v): {'a': 1, 'b': 2, 'c': 3, 'd': 2, 'e': 2, 'f': 1}


DUAL HYPERGRAPH

In [11]:
# PAPER: The dual hypergraph H* is defined by swapping vertices and hyperedges:
#   V(H*) = 𝓔(H)
#   For each vertex v ∈ V(H), we form a hyperedge E_v in H*:
#       E_v = { e ∈ 𝓔(H) : v ∈ e }
#
# In HyperNetX: dual() performs exactly this incidence transpose.

H_star = H.dual()
print_hyperedges(H_star, title="Hyperedges 𝓔(H*) (each corresponds to a vertex of H)")

Hyperedges 𝓔(H*) (each corresponds to a vertex of H)
  a: ['e1']
  b: ['e1', 'e2']
  c: ['e1', 'e2', 'e3']
  d: ['e1', 'e3']
  e: ['e3', 'e4']
  f: ['e4']


In [12]:
# SANITY CHECK: The dual of the dual should give us back the original hypergraph.
H_star_star = H_star.dual()

print("H == H**:", H == H_star_star)

H == H**: True


In [13]:
# ATTEMPTING SUBHYPERGRAPH CONSTRUCTION
# PAPER: A subhypergraph H' of H is defined by a subset of the hyperedges of H. Formally, if 𝓔' ⊆ 𝓔(H), then H' = (V', 𝓔') where V' is the union of the hyperedges in 𝓔'.

# PAPER: Restricting to a subfamily 𝓔' ⊆ 𝓔(H) gives a subhypergraph with
# the same incidence but fewer hyperedges.

H_sub = H.restrict_to_edges(["e1", "e3"])
print_hyperedges(H_sub, title="Subhypergraph with hyperedges {e1, e3}")

Subhypergraph with hyperedges {e1, e3}
  e1: ['a', 'b', 'c', 'd']
  e3: ['c', 'd', 'e']
